# TP4 - Synthèse FreeRTOS
## Utilisation des primitives du noyau temps réel

**ECUE : Atelier Systèmes Temps Réel**  
**Support matériel : Arduino AVR + FreeRTOS**

Ce notebook transforme le TP4 en cahier de conception. Les cellules Python décrivent les séquences attendues ; l'implémentation finale est à réaliser sur Arduino avec FreeRTOS.

## Objectifs

- concevoir une application temps réel complète ;
- synchroniser des tâches ;
- communiquer entre tâches ;
- partager des ressources ;
- gérer des interruptions ;
- mesurer le temps de changement de contexte.

Chaque réalisation doit préciser les tâches, les priorités, les périodes, les ressources partagées et les mécanismes de synchronisation utilisés.

## Exercice 1 - Chenillard

Un chenillard allume et éteint successivement les LED reliées aux broches Arduino **4, 5, 6, 7, 8 et 9**.

1. réaliser le montage ;
2. implémenter le mouvement avec des tâches périodiques indépendantes et une seule fonction paramétrable ;
3. implémenter ensuite le même mouvement avec des primitives FreeRTOS de synchronisation.

Une fonction de tâche peut recevoir le numéro de broche en paramètre :

```cpp
void vLedTask(void *parameter) {
    const uint8_t pin = (uint8_t)parameter;
    pinMode(pin, OUTPUT);
    for (;;) {
        digitalWrite(pin, HIGH);
        vTaskDelay(100 / portTICK_PERIOD_MS);
        digitalWrite(pin, LOW);
        vTaskDelay(500 / portTICK_PERIOD_MS);
    }
}
```

In [ ]:
BROCHES_LED = [4, 5, 6, 7, 8, 9]


def chenillard(broches, cycles=2, inverse=False, retour=False):
    """Retourne les broches activees dans l'ordre d'un chenillard."""
    if not broches:
        raise ValueError('La liste de broches ne peut pas etre vide.')
    sequence = broches[::-1] if inverse else list(broches)
    if retour and len(sequence) > 1:
        sequence = sequence + sequence[-2:0:-1]
    return [broche for _ in range(cycles) for broche in sequence]


assert set(chenillard(BROCHES_LED)) == set(BROCHES_LED)
assert chenillard(BROCHES_LED, cycles=1, inverse=True)[0] == 9
chenillard(BROCHES_LED, cycles=1, retour=True)

## Exercice 2 - Temps de changement de contexte

Écrire une application qui mesure le temps de changement de contexte sur la carte avec :

- `vTaskSuspend()` et `vTaskResume()` ;
- une synchronisation par sémaphore binaire.

Utiliser un oscilloscope. Basculer une sortie GPIO juste avant et juste après l'opération mesurée, puis calculer la durée à partir de la largeur de l'impulsion. Comparer les deux méthodes et documenter la fréquence du système et la configuration FreeRTOS.

In [ ]:
def temps_contexte(frequence_hz, ticks_mesures):
    return [ticks / frequence_hz * 1_000_000 for ticks in ticks_mesures]

# Exemple de conversion de ticks en microsecondes. Remplacer par les mesures réelles.
temps_contexte(1000, [1, 2, 3])

## Exercice 3 - Feu de croisement

Deux tâches contrôlent les deux axes : `EstOuest()` et `NordSud()`. Implémenter les fonctions :

- `Vert()` : vert allumé, autres feux éteints ;
- `Orange()` : jaune allumé, autres feux éteints ;
- `Rouge()` : rouge allumé, autres feux éteints.

Déclarer `LongDELAY` pour la durée du vert et `CourtDELAY` pour la durée de l'orange. La séquence habituelle est **Vert -> Orange -> Rouge**. Ajouter une interruption par bouton poussoir sur chaque axe afin de permettre la traversée des piétons en sécurité.

In [ ]:
sequence = [
    ('EstOuest', 'Vert', 'NordSud', 'Rouge'),
    ('EstOuest', 'Orange', 'NordSud', 'Rouge'),
    ('EstOuest', 'Rouge', 'NordSud', 'Vert'),
    ('EstOuest', 'Rouge', 'NordSud', 'Orange'),
]
for etat in sequence:
    print(f'{etat[0]}={etat[1]} | {etat[2]}={etat[3]}')

## Exercice 4 - Synchronisation de trois tâches

Les tâches `T1`, `T2` et `T3` exécutent chacune des cycles numérotés.

Contraintes :

- un cycle de `T1` s'exécute en concurrence avec un cycle de `T2` ;
- `T3` démarre un cycle uniquement lorsque `T1` et `T2` ont terminé ;
- après le cycle de `T3`, `T1` et `T2` recommencent ensemble.

Concevoir les sémaphores nécessaires. Un schéma classique utilise deux sémaphores de fin (`T1_termine`, `T2_termine`) et un mécanisme de rendez-vous pour autoriser `T3`, puis deux signaux de reprise pour le cycle suivant.

In [ ]:
def cycles_synchronises(nombre_cycles):
    journal = []
    for cycle in range(nombre_cycles):
        journal.extend([(cycle, 'T1'), (cycle, 'T2')])
        journal.append((cycle, 'T3', 'après T1 et T2'))
    return journal

cycles_synchronises(3)

## Compte rendu final

Pour chaque exercice, fournir le schéma de câblage, le code Arduino, la configuration des tâches et les observations. Pour les mesures, indiquer l'instrument utilisé, les unités, les répétitions et la valeur moyenne. Pour les synchronisations, justifier l'absence de conflit, de famine et de deadlock.